# Graph Laplacian Spectral Decomposition + SVD (Java Notebook)

This notebook ports SVD ideas from `mikdunn/dna-sequencing-and-imaging-svd` into Java and combines them with graph-Laplacian spectral methods for image tiles.

## Goals
1. Build Java SVD reconstruction utilities (rank and energy-cutoff styles).
2. Build graph Laplacian spectral low-pass reconstruction on image tiles.
3. Compare multiple FFT-like / low-rank / spectral analysis modes with mask-focused metrics.
4. Export stack comparison panels, per-method outputs, and rankings.

## Mathematical background (detailed, model-by-model)

### Notation
For each tile, let $X\in\mathbb{R}^{m\times n}$ be grayscale intensity, $M$ the mask image, and $R$ the reference reconstruction image. All compared maps are normalized to $[0,1]$ before metric scoring.

### 1) SVD Rank-$r$ reconstruction (fixed-rank model)
For
$$X = U\Sigma V^\top,\quad \Sigma=\mathrm{diag}(\sigma_1,\dots,\sigma_q),\ q=\min(m,n),$$
the rank-$r$ approximation is
$$X_r = U_{[:,1:r]}\Sigma_{1:r,1:r}V_{[:,1:r]}^\top.$$
In code, this is the `svdLowRank(..., fixedRank=r, energyCutoff=null, ...)` path (here $r=20$ for SVD baseline; $r=40$ for CNN map projection).

### 2) SVD Energy-cutoff reconstruction (adaptive-rank model)
Define cumulative spectral energy
$$E(r)=\frac{\sum_{i=1}^{r}\sigma_i^2}{\sum_{i=1}^{q}\sigma_i^2}.$$
Choose the smallest $r$ such that $E(r)\ge \tau$ (here $\tau=0.99$), then reconstruct with that rank. In code this is `svdLowRank(..., fixedRank=null, energyCutoff=0.99, ...)`.

### 3) CNN-style ConvBank feature map (before SVD projection)
Given normalized tile $Z\in[0,1]^{m\times n}$, we compute deterministic handcrafted channels:
- $G_3=\sqrt{(K_x^{3\times3}*Z)^2+(K_y^{3\times3}*Z)^2}$ (3x3 Sobel gradient magnitude)
- $G_5=\sqrt{(K_x^{3\times3}*B_5)^2+(K_y^{3\times3}*B_5)^2}$ where $B_5=K_g^{5\times5}*Z$ (5x5-smoothed gradient)
- $L=|K_\Delta*Z|$ (Laplacian magnitude)
- $D=|Z-B_5|$ (DoG-like residual)
- $T=\mathrm{LocalStd}_{3\times3}(Z)$ (local texture/contrast)
- $B=K_{\text{mean}}^{3\times3}*Z$ (local mean)

A blended feature map is then
$$F=\max\Big(0,\alpha F_{\text{base}}+(1-\alpha)F_{\text{multi}}+\eta\,(G_3+0.6L)\Big),$$
with
$$F_{\text{base}}=0.95G_3+0.35L+0.55B+0.30Z,$$
$$F_{\text{multi}}=0.80G_3+0.50G_5+0.30L+0.40D+0.25T+0.40B+0.45Z.$$
Then a gamma nonlinearity is applied after normalization:
$$\tilde F = \big(\mathrm{Norm01}(F)\big)^{\gamma}.$$

### 4) CNN map projection variants
After feature fusion, two SVD projection variants are compared:

**(a) CNN + Rank-40 SVD**
$$Y_{\text{cnn-r40}} = \mathrm{SVDRank}\big(\tilde F, 40\big).$$

**(b) CNN + Energy-99% SVD**
$$Y_{\text{cnn-e99}} = \mathrm{SVDEnergy}\big(\tilde F, 0.99\big).$$

Both are followed by mild unsharp post-processing (anti-fuzz):
$$Y = \mathrm{clip}_{[0,1]}\Big(Y_0 + \lambda\big(Y_0 - (K_{\text{mean}}^{3\times3}*Y_0)\big)\Big).$$
Here $\lambda$ is the `unsharpAmount` parameter.

### 5) Cached evaluator mathematics (why it is equivalent)
The cache stores per-tile deterministic channels $(G_3,G_5,L,D,T,B,Z)$. Parameter search only changes $(\alpha,\gamma,\eta,\lambda)$ and SVD projection choice, so
$$\text{cache}(X)\Rightarrow (G_3,G_5,L,D,T,B,Z)$$
does not alter the objective; it only avoids recomputing identical convolutions for each candidate. Thus the optimized score is mathematically identical to full recomputation.

### 6) Two-stage deterministic tuning objective
For each candidate parameter vector $\theta=(\alpha,\gamma,\eta,\lambda)$ we compute
$$\mathrm{sep}(\theta)=\mathbb{E}[T_\theta\mid M>t]-\mathbb{E}[T_\theta\mid M\le t],$$
$$\mathrm{corr}(\theta)=\mathrm{Pearson}(T_\theta,R),$$
$$J(\theta)=0.7\,\mathrm{sep}(\theta)+0.3\,\mathrm{corr}(\theta).$$

Search is deterministic:
1. **Coarse stage** over a fixed grid of $(\alpha,\gamma,\eta,\lambda)$.
2. **Refinement stage** over a local neighborhood around the coarse best.

The same two-stage process is run for both CNN projection heads (rank-40 and energy-99), so comparisons are fair.

### 7) Final metric used for ranking methods
For each method, per-tile scores are averaged then ranked by
$$\mathrm{score}=0.7\,\overline{\mathrm{sep}}+0.3\,\overline{\mathrm{corr}}.$$
Higher score indicates stronger mask contrast while preserving structural agreement with $R$.

## Citations (methods + biological/imaging use)
- Eckart, C., & Young, G. (1936). *The approximation of one matrix by another of lower rank*. Psychometrika.
- Chung, F. R. K. (1997). *Spectral Graph Theory*. CBMS.
- Fiedler, M. (1973). *Algebraic connectivity of graphs*. Czechoslovak Mathematical Journal.
- Belkin, M., & Niyogi, P. (2003). *Laplacian Eigenmaps for dimensionality reduction and data representation*. Neural Computation.
- Ng, A. Y., Jordan, M. I., & Weiss, Y. (2002). *On spectral clustering: Analysis and an algorithm*. NIPS.
- Zhou, D., Huang, J., & Schölkopf, B. (2006). *Learning with hypergraphs: Clustering, classification, and embedding*. NIPS.
- LeCun, Y., Bottou, L., Bengio, Y., & Haffner, P. (1998). *Gradient-based learning applied to document recognition*. Proc. IEEE.
- Manjón, J. V., et al. (2010). *Adaptive non-local means denoising of MR images*. Medical Image Analysis.
- Rajwade, A., et al. (2013). *Image denoising using local PCA and graph-based methods* (survey context).
- Rezakhaniha, R., et al. (2012). *Collagen waviness/orientation quantification in vascular tissue*. Biomech Model Mechanobiol.
- Bredfeldt, J. S., et al. (2014). *Collagen fiber segmentation from SHG microscopy*. J Biomed Opt.

Repository inspirations used for Java port logic:
- `src/imagesvd.py` (rank/energy SVD design)
- `src/microscopy_svd_case_study.py` (microscopy denoising case-study framing)
- `src/dim_reduction_demo.py` and `src/public_seq_pipeline.py` (spectral/Laplacian workflow context)

In [1]:
System.out.println("Java kernel is running ✅");
System.out.println("java.version: " + System.getProperty("java.version"));
System.out.println("user.dir:     " + System.getProperty("user.dir"));

Java kernel is running ✅
java.version: 25.0.2
user.dir:     c:\Users\dunnmk\repos\imgjplugin\notebooks


In [2]:
import java.awt.Color;
import java.awt.Font;
import java.awt.Graphics2D;
import java.awt.RenderingHints;
import java.awt.image.BufferedImage;
import javax.imageio.ImageIO;
import java.io.File;
import java.nio.charset.StandardCharsets;
import java.nio.file.*;
import java.util.ArrayList;
import java.util.Arrays;
import java.util.Comparator;
import java.util.HashMap;
import java.util.List;
import java.util.Locale;
import java.util.Map;
import java.util.regex.Matcher;
import java.util.regex.Pattern;

record MethodCfg(String label, String key) {}
record MethodScore(String label, double sepMean, double corrMean, double score, int nTiles) {}

Path findProjectRoot(Path start) {
    Path p = start.toAbsolutePath().normalize();
    for (int i = 0; i < 12 && p != null; i++) {
        if (Files.exists(p.resolve("FFT").resolve("pom.xml"))) return p;
        p = p.getParent();
    }
    return null;
}

double[][] gray(BufferedImage img) {
    int h = img.getHeight(), w = img.getWidth();
    double[][] out = new double[h][w];
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            int rgb = img.getRGB(x, y);
            int r = (rgb >> 16) & 0xff;
            int g = (rgb >> 8) & 0xff;
            int b = rgb & 0xff;
            out[y][x] = 0.299 * r + 0.587 * g + 0.114 * b;
        }
    }
    return out;
}

double[][] normalize01(double[][] a) {
    int h = a.length, w = a[0].length;
    double min = Double.POSITIVE_INFINITY, max = Double.NEGATIVE_INFINITY;
    for (double[] row : a) for (double v : row) { min = Math.min(min, v); max = Math.max(max, v); }
    double span = Math.max(1e-9, max - min);
    double[][] out = new double[h][w];
    for (int y = 0; y < h; y++) for (int x = 0; x < w; x++) out[y][x] = (a[y][x] - min) / span;
    return out;
}

BufferedImage toGrayImage01(double[][] a) {
    double[][] n = normalize01(a);
    int h = n.length, w = n[0].length;
    BufferedImage out = new BufferedImage(w, h, BufferedImage.TYPE_BYTE_GRAY);
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            int v = (int)Math.round(255.0 * n[y][x]);
            v = Math.max(0, Math.min(255, v));
            int rgb = (v << 16) | (v << 8) | v;
            out.setRGB(x, y, rgb);
        }
    }
    return out;
}

double pearson2D(double[][] a, double[][] b) {
    int h = Math.min(a.length, b.length);
    int w = Math.min(a[0].length, b[0].length);
    int n = h * w;
    double sa = 0.0, sb = 0.0;
    for (int y = 0; y < h; y++) for (int x = 0; x < w; x++) { sa += a[y][x]; sb += b[y][x]; }
    double ma = sa / n, mb = sb / n;
    double num = 0.0, da = 0.0, db = 0.0;
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            double xa = a[y][x] - ma;
            double xb = b[y][x] - mb;
            num += xa * xb;
            da += xa * xa;
            db += xb * xb;
        }
    }
    return num / Math.sqrt(Math.max(1e-12, da * db));
}

double[] inOutMeans(double[][] map01, double[][] maskGray) {
    int h = Math.min(map01.length, maskGray.length);
    int w = Math.min(map01[0].length, maskGray[0].length);
    double in = 0.0, out = 0.0;
    int nin = 0, nout = 0;
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            if (maskGray[y][x] > 32.0) { in += map01[y][x]; nin++; }
            else { out += map01[y][x]; nout++; }
        }
    }
    return new double[] { nin == 0 ? 0 : in / nin, nout == 0 ? 0 : out / nout };
}

double[][] copy2D(double[][] a) {
    int h = a.length, w = a[0].length;
    double[][] out = new double[h][w];
    for (int y = 0; y < h; y++) System.arraycopy(a[y], 0, out[y], 0, w);
    return out;
}

double[][] resizeNN(double[][] src, int outH, int outW) {
    int h = src.length, w = src[0].length;
    double[][] out = new double[outH][outW];
    for (int y = 0; y < outH; y++) {
        int sy = Math.min(h - 1, (int)Math.floor(y * (h / (double)outH)));
        for (int x = 0; x < outW; x++) {
            int sx = Math.min(w - 1, (int)Math.floor(x * (w / (double)outW)));
            out[y][x] = src[sy][sx];
        }
    }
    return out;
}

System.out.println("Helpers loaded ✅");

Helpers loaded ✅


In [3]:
// ---------- SVD port (inspired by src/imagesvd.py) ----------
double dot(double[] a, double[] b) {
    double s = 0.0;
    for (int i = 0; i < a.length; i++) s += a[i] * b[i];
    return s;
}

double norm2(double[] v) {
    return Math.sqrt(Math.max(1e-18, dot(v, v)));
}

void normalize(double[] v) {
    double n = norm2(v);
    for (int i = 0; i < v.length; i++) v[i] /= n;
}

double[] matVec(double[][] A, double[] x) {
    int m = A.length, n = A[0].length;
    double[] y = new double[m];
    for (int i = 0; i < m; i++) {
        double s = 0.0;
        for (int j = 0; j < n; j++) s += A[i][j] * x[j];
        y[i] = s;
    }
    return y;
}

double[][] transpose(double[][] A) {
    int m = A.length, n = A[0].length;
    double[][] T = new double[n][m];
    for (int i = 0; i < m; i++) for (int j = 0; j < n; j++) T[j][i] = A[i][j];
    return T;
}

double[][] matMul(double[][] A, double[][] B) {
    int m = A.length, k = A[0].length, n = B[0].length;
    double[][] C = new double[m][n];
    for (int i = 0; i < m; i++) {
        for (int t = 0; t < k; t++) {
            double a = A[i][t];
            for (int j = 0; j < n; j++) C[i][j] += a * B[t][j];
        }
    }
    return C;
}

double[][] covarianceXXT(double[][] X) {
    int m = X.length, n = X[0].length;
    double[][] C = new double[m][m];
    for (int i = 0; i < m; i++) {
        for (int j = i; j < m; j++) {
            double s = 0.0;
            for (int t = 0; t < n; t++) s += X[i][t] * X[j][t];
            C[i][j] = s;
            C[j][i] = s;
        }
    }
    return C;
}

record EigPair(double lambda, double[] vec) {}

EigPair powerIterationSymmetric(double[][] A, List<double[]> orth, int maxIter, double tol) {
    int n = A.length;
    java.util.Random rng = new java.util.Random(42 + orth.size());
    double[] v = new double[n];
    for (int i = 0; i < n; i++) v[i] = rng.nextDouble() - 0.5;
    normalize(v);

    for (int it = 0; it < maxIter; it++) {
        double[] w = matVec(A, v);
        for (double[] q : orth) {
            double c = dot(w, q);
            for (int i = 0; i < n; i++) w[i] -= c * q[i];
        }
        double nw = norm2(w);
        if (nw < 1e-12) break;
        for (int i = 0; i < n; i++) w[i] /= nw;

        double diff = 0.0;
        for (int i = 0; i < n; i++) {
            double d = w[i] - v[i];
            diff += d * d;
        }
        v = w;
        if (Math.sqrt(diff) < tol) break;
    }

    double[] Av = matVec(A, v);
    double lambda = dot(v, Av);
    return new EigPair(lambda, v);
}

int chooseRankFromEnergy(double[] s, double energy) {
    if (!(energy > 0.0 && energy <= 1.0)) throw new IllegalArgumentException("energy must be in (0,1]");
    double total = 0.0;
    for (double v : s) total += v * v;
    double cum = 0.0;
    for (int i = 0; i < s.length; i++) {
        cum += s[i] * s[i];
        if (cum / Math.max(1e-18, total) >= energy) return i + 1;
    }
    return s.length;
}

record SVDResult(double[][] recon, double[] singularValues, int rank, double energyAtRank) {}

SVDResult svdLowRank(double[][] X, Integer fixedRank, Double energyCutoff, int maxRankCap) {
    if ((fixedRank == null) == (energyCutoff == null)) throw new IllegalArgumentException("Provide exactly one of fixedRank or energyCutoff");

    int m = X.length, n = X[0].length;
    int kMax = Math.min(Math.min(m, n), maxRankCap);
    double[][] C = covarianceXXT(X);

    List<double[]> Ucols = new ArrayList<>();
    double[] sVals = new double[kMax];

    for (int k = 0; k < kMax; k++) {
        EigPair ep = powerIterationSymmetric(C, Ucols, 300, 1e-7);
        double lambda = Math.max(0.0, ep.lambda());
        if (lambda < 1e-10) {
            sVals = Arrays.copyOf(sVals, k);
            break;
        }
        Ucols.add(ep.vec());
        sVals[k] = Math.sqrt(lambda);
    }

    int found = Ucols.size();
    if (found == 0) return new SVDResult(copy2D(X), new double[] {0.0}, 1, 1.0);
    if (sVals.length != found) sVals = Arrays.copyOf(sVals, found);

    int r = (fixedRank != null) ? Math.max(1, Math.min(fixedRank, found)) : chooseRankFromEnergy(sVals, energyCutoff);

    double[][] U = new double[m][r];
    for (int j = 0; j < r; j++) {
        double[] u = Ucols.get(j);
        for (int i = 0; i < m; i++) U[i][j] = u[i];
    }

    double[][] Ut = transpose(U);
    double[][] Vt = new double[r][n];
    for (int j = 0; j < r; j++) {
        double sj = Math.max(1e-12, sVals[j]);
        for (int col = 0; col < n; col++) {
            double acc = 0.0;
            for (int i = 0; i < m; i++) acc += Ut[j][i] * X[i][col];
            Vt[j][col] = acc / sj;
        }
    }

    double[][] recon = new double[m][n];
    for (int i = 0; i < m; i++) {
        for (int col = 0; col < n; col++) {
            double s = 0.0;
            for (int j = 0; j < r; j++) s += U[i][j] * sVals[j] * Vt[j][col];
            recon[i][col] = s;
        }
    }

    double total = 0.0, keep = 0.0;
    for (int i = 0; i < found; i++) {
        total += sVals[i] * sVals[i];
        if (i < r) keep += sVals[i] * sVals[i];
    }
    double e = keep / Math.max(1e-18, total);

    return new SVDResult(recon, sVals, r, e);
}

System.out.println("SVD functions loaded ✅");

SVD functions loaded ✅


In [31]:
// ---------- Graph/Hypergraph helpers + improved CNN-style high-rank baseline ----------
record GraphSpecResult(double[][] recon, int kModes, double fiedlerNumber, double fiedlerStd) {}
record SymEig(double[] values, double[][] vectors) {}
record CnnTuneParams(double baseBlend, double gamma, double edgeBoost, double unsharpAmount) {}
record CnnTileFeatures(double[][] grad3, double[][] grad5, double[][] lapAbs, double[][] dog, double[][] tex, double[][] mean3, double[][] z) {}

int idx(int y, int x, int w) { return y * w + x; }

double[][] identity(int n) {
    double[][] I = new double[n][n];
    for (int i = 0; i < n; i++) I[i][i] = 1.0;
    return I;
}

SymEig jacobiSymmetric(double[][] Ain, int maxSweeps, double tol) {
    int n = Ain.length;
    double[][] A = new double[n][n];
    for (int i = 0; i < n; i++) System.arraycopy(Ain[i], 0, A[i], 0, n);
    double[][] V = identity(n);

    for (int sweep = 0; sweep < maxSweeps; sweep++) {
        int p = 0, q = 1;
        double maxOff = 0.0;
        for (int i = 0; i < n; i++) {
            for (int j = i + 1; j < n; j++) {
                double v = Math.abs(A[i][j]);
                if (v > maxOff) { maxOff = v; p = i; q = j; }
            }
        }
        if (maxOff < tol) break;

        double app = A[p][p], aqq = A[q][q], apq = A[p][q];
        double phi = 0.5 * Math.atan2(2.0 * apq, (aqq - app));
        double c = Math.cos(phi), s = Math.sin(phi);

        for (int k = 0; k < n; k++) {
            double akp = A[k][p], akq = A[k][q];
            A[k][p] = c * akp - s * akq;
            A[k][q] = s * akp + c * akq;
        }
        for (int k = 0; k < n; k++) {
            double apk = A[p][k], aqk = A[q][k];
            A[p][k] = c * apk - s * aqk;
            A[q][k] = s * apk + c * aqk;
        }

        A[p][q] = 0.0;
        A[q][p] = 0.0;

        for (int k = 0; k < n; k++) {
            double vkp = V[k][p], vkq = V[k][q];
            V[k][p] = c * vkp - s * vkq;
            V[k][q] = s * vkp + c * vkq;
        }
    }

    double[] evals = new double[n];
    for (int i = 0; i < n; i++) evals[i] = A[i][i];

    for (int i = 0; i < n - 1; i++) {
        int best = i;
        for (int j = i + 1; j < n; j++) if (evals[j] < evals[best]) best = j;
        if (best != i) {
            double te = evals[i]; evals[i] = evals[best]; evals[best] = te;
            for (int r = 0; r < n; r++) {
                double tv = V[r][i]; V[r][i] = V[r][best]; V[r][best] = tv;
            }
        }
    }
    return new SymEig(evals, V);
}

double std(double[] x) {
    double m = 0.0;
    for (double v : x) m += v;
    m /= Math.max(1, x.length);
    double s = 0.0;
    for (double v : x) { double d = v - m; s += d * d; }
    return Math.sqrt(s / Math.max(1, x.length));
}

double[][] buildSymmetricAffinity(double[][] img01, double sigmaI, boolean use8Nbr, double selfLoop) {
    int h = img01.length, w = img01[0].length;
    int n = h * w;
    double[][] W = new double[n][n];
    int[][] dirs4 = new int[][] {{1,0},{-1,0},{0,1},{0,-1}};
    int[][] dirs8 = new int[][] {{1,0},{-1,0},{0,1},{0,-1},{1,1},{1,-1},{-1,1},{-1,-1}};
    int[][] dirs = use8Nbr ? dirs8 : dirs4;

    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            int i = idx(y, x, w);
            W[i][i] = selfLoop;
            for (int[] d : dirs) {
                int yy = y + d[0], xx = x + d[1];
                if (yy < 0 || yy >= h || xx < 0 || xx >= w) continue;
                int j = idx(yy, xx, w);
                double di = img01[y][x] - img01[yy][xx];
                double wij = Math.exp(-(di * di) / Math.max(1e-9, sigmaI * sigmaI));
                if (wij > W[i][j]) {
                    W[i][j] = wij;
                    W[j][i] = wij;
                }
            }
        }
    }
    return W;
}

double[][] normalizedLaplacianFromW(double[][] W) {
    int n = W.length;
    double[] deg = new double[n];
    for (int i = 0; i < n; i++) {
        double s = 0.0;
        for (int j = 0; j < n; j++) s += W[i][j];
        deg[i] = s;
    }
    double[][] Ln = new double[n][n];
    for (int i = 0; i < n; i++) {
        Ln[i][i] = 1.0;
        double di = 1.0 / Math.sqrt(Math.max(1e-12, deg[i]));
        for (int j = 0; j < n; j++) {
            double dj = 1.0 / Math.sqrt(Math.max(1e-12, deg[j]));
            Ln[i][j] -= di * W[i][j] * dj;
        }
    }
    return Ln;
}

GraphSpecResult spectralLowPassFromLaplacian(double[][] Ln, double[][] signal2D, int kModes) {
    int h = signal2D.length, w = signal2D[0].length;
    int n = h * w;
    SymEig eig = jacobiSymmetric(Ln, 45, 1e-5);
    double[] evals = eig.values();
    double[][] V = eig.vectors();

    int k = Math.max(2, Math.min(kModes, n));
    double[] f = new double[n];
    for (int y = 0; y < h; y++) for (int x = 0; x < w; x++) f[idx(y,x,w)] = signal2D[y][x];

    double[] recon = new double[n];
    for (int mode = 0; mode < k; mode++) {
        double c = 0.0;
        for (int i = 0; i < n; i++) c += f[i] * V[i][mode];
        for (int i = 0; i < n; i++) recon[i] += c * V[i][mode];
    }

    double[][] rs = new double[h][w];
    for (int y = 0; y < h; y++) for (int x = 0; x < w; x++) rs[y][x] = recon[idx(y,x,w)];

    double lambda2 = (evals.length > 1) ? evals[1] : Double.NaN;
    double[] fvec = new double[n];
    if (evals.length > 1) for (int i = 0; i < n; i++) fvec[i] = V[i][1];
    double fstd = std(fvec);
    return new GraphSpecResult(rs, k, lambda2, fstd);
}

double[][] conv3x3(double[][] src, double[][] k) {
    int h = src.length, w = src[0].length;
    double[][] out = new double[h][w];
    for (int y = 1; y < h - 1; y++) {
        for (int x = 1; x < w - 1; x++) {
            double s = 0.0;
            for (int j = -1; j <= 1; j++) for (int i = -1; i <= 1; i++) s += src[y + j][x + i] * k[j + 1][i + 1];
            out[y][x] = s;
        }
    }
    return out;
}

double[][] convNxN(double[][] src, double[][] k) {
    int h = src.length, w = src[0].length;
    int kh = k.length, kw = k[0].length;
    int cy = kh / 2, cx = kw / 2;
    double[][] out = new double[h][w];
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            double s = 0.0;
            for (int j = 0; j < kh; j++) {
                int yy = Math.max(0, Math.min(h - 1, y + j - cy));
                for (int i = 0; i < kw; i++) {
                    int xx = Math.max(0, Math.min(w - 1, x + i - cx));
                    s += src[yy][xx] * k[j][i];
                }
            }
            out[y][x] = s;
        }
    }
    return out;
}

double[][] localStd3x3(double[][] src) {
    int h = src.length, w = src[0].length;
    double[][] out = new double[h][w];
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            double mean = 0.0;
            int n = 0;
            for (int j = -1; j <= 1; j++) {
                int yy = Math.max(0, Math.min(h - 1, y + j));
                for (int i = -1; i <= 1; i++) {
                    int xx = Math.max(0, Math.min(w - 1, x + i));
                    mean += src[yy][xx];
                    n++;
                }
            }
            mean /= Math.max(1, n);
            double var = 0.0;
            for (int j = -1; j <= 1; j++) {
                int yy = Math.max(0, Math.min(h - 1, y + j));
                for (int i = -1; i <= 1; i++) {
                    int xx = Math.max(0, Math.min(w - 1, x + i));
                    double d = src[yy][xx] - mean;
                    var += d * d;
                }
            }
            out[y][x] = Math.sqrt(var / Math.max(1, n));
        }
    }
    return out;
}

CnnTileFeatures extractCnnTileFeatures(double[][] img) {
    double[][] z = normalize01(img);

    double[][] kx = new double[][] {{-1,0,1},{-2,0,2},{-1,0,1}};
    double[][] ky = new double[][] {{-1,-2,-1},{0,0,0},{1,2,1}};
    double[][] kl = new double[][] {{0,1,0},{1,-4,1},{0,1,0}};
    double[][] kb = new double[][] {{1/9.0,1/9.0,1/9.0},{1/9.0,1/9.0,1/9.0},{1/9.0,1/9.0,1/9.0}};
    double[][] kg5 = new double[][] {
        {1/273.0, 4/273.0, 7/273.0, 4/273.0, 1/273.0},
        {4/273.0,16/273.0,26/273.0,16/273.0, 4/273.0},
        {7/273.0,26/273.0,41/273.0,26/273.0, 7/273.0},
        {4/273.0,16/273.0,26/273.0,16/273.0, 4/273.0},
        {1/273.0, 4/273.0, 7/273.0, 4/273.0, 1/273.0}
    };

    double[][] gx3 = conv3x3(z, kx);
    double[][] gy3 = conv3x3(z, ky);
    double[][] lap = conv3x3(z, kl);
    double[][] mean3 = conv3x3(z, kb);
    double[][] blur5 = convNxN(z, kg5);
    double[][] gx5 = conv3x3(blur5, kx);
    double[][] gy5 = conv3x3(blur5, ky);
    double[][] tex = localStd3x3(z);

    int h = z.length, w = z[0].length;
    double[][] grad3 = new double[h][w];
    double[][] grad5 = new double[h][w];
    double[][] lapAbs = new double[h][w];
    double[][] dog = new double[h][w];
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            grad3[y][x] = Math.sqrt(gx3[y][x] * gx3[y][x] + gy3[y][x] * gy3[y][x]);
            grad5[y][x] = Math.sqrt(gx5[y][x] * gx5[y][x] + gy5[y][x] * gy5[y][x]);
            lapAbs[y][x] = Math.abs(lap[y][x]);
            dog[y][x] = Math.abs(z[y][x] - blur5[y][x]);
        }
    }

    return new CnnTileFeatures(grad3, grad5, lapAbs, dog, tex, mean3, z);
}

double[][] cnnMapFromFeatures(CnnTileFeatures f, int rank, CnnTuneParams p) {
    int h = f.z().length, w = f.z()[0].length;
    double[][] feat = new double[h][w];
    double bBase = p.baseBlend();
    double bMulti = 1.0 - bBase;
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            double vBase = 0.95 * f.grad3()[y][x] + 0.35 * f.lapAbs()[y][x] + 0.55 * f.mean3()[y][x] + 0.30 * f.z()[y][x];
            double vMulti = 0.80 * f.grad3()[y][x]
                          + 0.50 * f.grad5()[y][x]
                          + 0.30 * f.lapAbs()[y][x]
                          + 0.40 * f.dog()[y][x]
                          + 0.25 * f.tex()[y][x]
                          + 0.40 * f.mean3()[y][x]
                          + 0.45 * f.z()[y][x];
            double edgePreserve = p.edgeBoost() * (f.grad3()[y][x] + 0.6 * f.lapAbs()[y][x]);
            feat[y][x] = Math.max(0.0, bBase * vBase + bMulti * vMulti + edgePreserve);
        }
    }

    double[][] feat01 = normalize01(feat);
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            feat01[y][x] = Math.pow(Math.max(0.0, Math.min(1.0, feat01[y][x])), p.gamma());
        }
    }

    double[][] hi = svdLowRank(feat01, Math.max(30, rank), null, Math.max(80, rank + 20)).recon();
    double[][] hi01 = normalize01(hi);

    double[][] kb = new double[][] {{1/9.0,1/9.0,1/9.0},{1/9.0,1/9.0,1/9.0},{1/9.0,1/9.0,1/9.0}};
    // anti-fuzz step: mild unsharp mask to preserve line crispness
    double[][] blurHi = conv3x3(hi01, kb);
    double[][] sharp = new double[h][w];
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            double u = hi01[y][x] + p.unsharpAmount() * (hi01[y][x] - blurHi[y][x]);
            sharp[y][x] = Math.max(0.0, Math.min(1.0, u));
        }
    }
    return sharp;
}

double[][] cnnMapFromFeaturesEnergy99(CnnTileFeatures f, CnnTuneParams p) {
    int h = f.z().length, w = f.z()[0].length;
    double[][] feat = new double[h][w];
    double bBase = p.baseBlend();
    double bMulti = 1.0 - bBase;
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            double vBase = 0.95 * f.grad3()[y][x] + 0.35 * f.lapAbs()[y][x] + 0.55 * f.mean3()[y][x] + 0.30 * f.z()[y][x];
            double vMulti = 0.80 * f.grad3()[y][x]
                          + 0.50 * f.grad5()[y][x]
                          + 0.30 * f.lapAbs()[y][x]
                          + 0.40 * f.dog()[y][x]
                          + 0.25 * f.tex()[y][x]
                          + 0.40 * f.mean3()[y][x]
                          + 0.45 * f.z()[y][x];
            double edgePreserve = p.edgeBoost() * (f.grad3()[y][x] + 0.6 * f.lapAbs()[y][x]);
            feat[y][x] = Math.max(0.0, bBase * vBase + bMulti * vMulti + edgePreserve);
        }
    }

    double[][] feat01 = normalize01(feat);
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            feat01[y][x] = Math.pow(Math.max(0.0, Math.min(1.0, feat01[y][x])), p.gamma());
        }
    }

    double[][] hi = svdLowRank(feat01, null, 0.99, 120).recon();
    double[][] hi01 = normalize01(hi);

    double[][] kb = new double[][] {{1/9.0,1/9.0,1/9.0},{1/9.0,1/9.0,1/9.0},{1/9.0,1/9.0,1/9.0}};
    double[][] blurHi = conv3x3(hi01, kb);
    double[][] sharp = new double[h][w];
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            double u = hi01[y][x] + p.unsharpAmount() * (hi01[y][x] - blurHi[y][x]);
            sharp[y][x] = Math.max(0.0, Math.min(1.0, u));
        }
    }
    return sharp;
}

double[][] cnnStyleHighRankWithParams(double[][] img, int rank, CnnTuneParams p) {
    return cnnMapFromFeatures(extractCnnTileFeatures(img), rank, p);
}

double[][] cnnStyleHighRank(double[][] img, int rank) {
    return cnnMapFromFeatures(extractCnnTileFeatures(img), rank, new CnnTuneParams(0.60, 1.05, 0.18, 0.20));
}

System.out.println("Graph/hypergraph helpers + tuned CNN controls loaded ✅");

Graph/hypergraph helpers + tuned CNN controls loaded ✅


In [33]:
// ---------- Apply top-3 methods only + deterministic CNN two-stage search ----------
record TopMethodOutput(double[][] map) {}
record TileDatum(String base, Path dir, int id, double[][] g, double[][] maskG, double[][] rec01, CnnTileFeatures cnnFeat) {}
record CnnEval(CnnTuneParams params, double sepMean, double corrMean, double score, int nTiles) {}

TopMethodOutput methodMap(double[][] g, String methodKey, CnnTuneParams cnnParams) {
    if ("svd_rank20".equals(methodKey)) {
        return new TopMethodOutput(svdLowRank(g, 20, null, 60).recon());
    }
    if ("svd_energy99".equals(methodKey)) {
        return new TopMethodOutput(svdLowRank(g, null, 0.99, 60).recon());
    }
    if ("cnn_rank40".equals(methodKey)) {
        return new TopMethodOutput(cnnStyleHighRankWithParams(g, 40, cnnParams));
    }
    throw new IllegalArgumentException("Unknown method: " + methodKey);
}

double clamp(double v, double lo, double hi) {
    return Math.max(lo, Math.min(hi, v));
}

double round2(double v) {
    return Math.round(v * 100.0) / 100.0;
}

Path root = findProjectRoot(Paths.get(System.getProperty("user.dir")));
if (root == null) throw new RuntimeException("Could not locate project root");

List<String> bases = Arrays.asList("C15D5P001_1", "Picture1");
List<Path> dirs = Arrays.asList(
    root.resolve("notebooks").resolve("_assets").resolve("fiba_wavelet_montage").resolve("generated_from_source"),
    root.resolve("notebooks").resolve("_assets").resolve("fiba_wavelet_montage").resolve("generated_from_source_picture1")
);

List<MethodCfg> methods = Arrays.asList(
    new MethodCfg("SVD Rank-20", "svd_rank20"),
    new MethodCfg("SVD Energy-99%", "svd_energy99"),
    new MethodCfg("CNN-style ConvBank + SVD Rank-40", "cnn_rank40")
);

// 1) Load tiles once (deterministic ordering)
List<TileDatum> tiles = new ArrayList<>();
for (int ds = 0; ds < bases.size(); ds++) {
    String base = bases.get(ds);
    Path dir = dirs.get(ds);

    Pattern p = Pattern.compile("^" + Pattern.quote(base) + "_tile(\\d+)_crop\\.jpg$");
    List<Integer> ids = new ArrayList<>();
    try (DirectoryStream<Path> stream = Files.newDirectoryStream(dir, "*_crop.jpg")) {
        for (Path f : stream) {
            Matcher m = p.matcher(f.getFileName().toString());
            if (m.matches()) ids.add(Integer.parseInt(m.group(1)));
        }
    }
    ids.sort(Comparator.naturalOrder());

    for (int id : ids) {
        Path cropPath = dir.resolve(base + "_tile" + id + "_crop.jpg");
        Path maskPath = dir.resolve(base + "_tile" + id + "_mask.jpg");
        Path recPath = dir.resolve(base + "_tile" + id + "_rec.jpg");
        if (!Files.isRegularFile(cropPath) || !Files.isRegularFile(maskPath) || !Files.isRegularFile(recPath)) continue;

        BufferedImage crop = ImageIO.read(cropPath.toFile());
        BufferedImage mask = ImageIO.read(maskPath.toFile());
        BufferedImage rec = ImageIO.read(recPath.toFile());
        if (crop == null || mask == null || rec == null) continue;

        double[][] gTile = gray(crop);
        tiles.add(new TileDatum(
            base,
            dir,
            id,
            gTile,
            gray(mask),
            normalize01(gray(rec)),
            extractCnnTileFeatures(gTile)
        ));
    }
}
if (tiles.isEmpty()) throw new RuntimeException("No tiles loaded for scoring.");
System.out.println("Loaded tiles for benchmarking: " + tiles.size());

// 2) Deterministic two-stage CNN search over objective: 0.7*sep + 0.3*corr
List<CnnEval> evals = new ArrayList<>();
Map<String, Boolean> seen = new HashMap<>();

java.util.function.Consumer<CnnTuneParams> evalOne = (pIn) -> {
    CnnTuneParams p = new CnnTuneParams(
        round2(clamp(pIn.baseBlend(), 0.45, 0.75)),
        round2(clamp(pIn.gamma(), 0.85, 1.15)),
        round2(clamp(pIn.edgeBoost(), 0.00, 0.35)),
        round2(clamp(pIn.unsharpAmount(), 0.00, 0.35))
    );
    String key = String.format(Locale.US, "%.2f|%.2f|%.2f|%.2f", p.baseBlend(), p.gamma(), p.edgeBoost(), p.unsharpAmount());
    if (seen.containsKey(key)) return;
    seen.put(key, true);

    double sepSum = 0.0, corrSum = 0.0;
    int n = 0;
    for (TileDatum t : tiles) {
        double[][] map01 = normalize01(cnnMapFromFeatures(t.cnnFeat(), 40, p));
        double[] io = inOutMeans(map01, t.maskG());
        sepSum += (io[0] - io[1]);
        corrSum += pearson2D(map01, t.rec01());
        n++;
    }
    if (n > 0) {
        double sepMean = sepSum / n;
        double corrMean = corrSum / n;
        double score = 0.7 * sepMean + 0.3 * corrMean;
        evals.add(new CnnEval(p, sepMean, corrMean, score, n));
    }
};

// Stage A: coarse search
double[] coarseBaseBlend = new double[] {0.55, 0.65};
double[] coarseGamma = new double[] {0.95, 1.05};
double[] coarseEdge = new double[] {0.10, 0.26};
double[] coarseUnsharp = new double[] {0.12, 0.28};

for (double bb : coarseBaseBlend) {
    for (double gm : coarseGamma) {
        for (double eb : coarseEdge) {
            for (double us : coarseUnsharp) {
                evalOne.accept(new CnnTuneParams(bb, gm, eb, us));
            }
        }
    }
}

evals.sort((a, b) -> {
    int c = Double.compare(b.score(), a.score());
    if (c != 0) return c;
    return Double.compare(b.corrMean(), a.corrMean());
});
CnnEval coarseBest = evals.get(0);
CnnTuneParams cb = coarseBest.params();
System.out.printf(Locale.US,
    "Stage A (coarse) best: baseBlend=%.2f gamma=%.2f edgeBoost=%.2f unsharp=%.2f | score=%.5f%n",
    cb.baseBlend(), cb.gamma(), cb.edgeBoost(), cb.unsharpAmount(), coarseBest.score()
 );

// Stage B: local refinement around coarse best (3x3x3x3 neighborhood)
double dBase = 0.03, dGamma = 0.03, dEdge = 0.04, dUnsharp = 0.04;
double[] steps = new double[] {-1.0, 0.0, 1.0};
for (double s1 : steps) {
    for (double s2 : steps) {
        for (double s3 : steps) {
            for (double s4 : steps) {
                evalOne.accept(new CnnTuneParams(
                    cb.baseBlend() + s1 * dBase,
                    cb.gamma() + s2 * dGamma,
                    cb.edgeBoost() + s3 * dEdge,
                    cb.unsharpAmount() + s4 * dUnsharp
                ));
            }
        }
    }
}

evals.sort((a, b) -> {
    int c = Double.compare(b.score(), a.score());
    if (c != 0) return c;
    return Double.compare(b.corrMean(), a.corrMean());
});

CnnEval bestCnn = evals.get(0);
CnnTuneParams bestParams = bestCnn.params();
System.out.println("Total unique CNN candidates evaluated: " + evals.size());
System.out.printf(Locale.US,
    "Stage B (refined) best: baseBlend=%.2f, gamma=%.2f, edgeBoost=%.2f, unsharp=%.2f | sep=%.5f corr=%.5f score=%.5f%n",
    bestParams.baseBlend(), bestParams.gamma(), bestParams.edgeBoost(), bestParams.unsharpAmount(),
    bestCnn.sepMean(), bestCnn.corrMean(), bestCnn.score()
 );

// Dedicated two-stage tuning for CNN-style ConvBank + SVD Energy-99%
List<CnnEval> evalsE99 = new ArrayList<>();
Map<String, Boolean> seenE99 = new HashMap<>();

java.util.function.Consumer<CnnTuneParams> evalOneE99 = (pIn) -> {
    CnnTuneParams p = new CnnTuneParams(
        round2(clamp(pIn.baseBlend(), 0.45, 0.75)),
        round2(clamp(pIn.gamma(), 0.85, 1.15)),
        round2(clamp(pIn.edgeBoost(), 0.00, 0.35)),
        round2(clamp(pIn.unsharpAmount(), 0.00, 0.35))
    );
    String key = String.format(Locale.US, "%.2f|%.2f|%.2f|%.2f", p.baseBlend(), p.gamma(), p.edgeBoost(), p.unsharpAmount());
    if (seenE99.containsKey(key)) return;
    seenE99.put(key, true);

    double sepSum = 0.0, corrSum = 0.0;
    int n = 0;
    for (TileDatum t : tiles) {
        double[][] map01 = normalize01(cnnMapFromFeaturesEnergy99(t.cnnFeat(), p));
        double[] io = inOutMeans(map01, t.maskG());
        sepSum += (io[0] - io[1]);
        corrSum += pearson2D(map01, t.rec01());
        n++;
    }
    if (n > 0) {
        double sepMean = sepSum / n;
        double corrMean = corrSum / n;
        double score = 0.7 * sepMean + 0.3 * corrMean;
        evalsE99.add(new CnnEval(p, sepMean, corrMean, score, n));
    }
};

for (double bb : coarseBaseBlend) {
    for (double gm : coarseGamma) {
        for (double eb : coarseEdge) {
            for (double us : coarseUnsharp) {
                evalOneE99.accept(new CnnTuneParams(bb, gm, eb, us));
            }
        }
    }
}

evalsE99.sort((a, b) -> {
    int c = Double.compare(b.score(), a.score());
    if (c != 0) return c;
    return Double.compare(b.corrMean(), a.corrMean());
});
CnnEval coarseBestE99 = evalsE99.get(0);
CnnTuneParams cbE99 = coarseBestE99.params();
System.out.printf(Locale.US,
    "E99 Stage A (coarse) best: baseBlend=%.2f gamma=%.2f edgeBoost=%.2f unsharp=%.2f | score=%.5f%n",
    cbE99.baseBlend(), cbE99.gamma(), cbE99.edgeBoost(), cbE99.unsharpAmount(), coarseBestE99.score()
 );

for (double s1 : steps) {
    for (double s2 : steps) {
        for (double s3 : steps) {
            for (double s4 : steps) {
                evalOneE99.accept(new CnnTuneParams(
                    cbE99.baseBlend() + s1 * dBase,
                    cbE99.gamma() + s2 * dGamma,
                    cbE99.edgeBoost() + s3 * dEdge,
                    cbE99.unsharpAmount() + s4 * dUnsharp
                ));
            }
        }
    }
}

evalsE99.sort((a, b) -> {
    int c = Double.compare(b.score(), a.score());
    if (c != 0) return c;
    return Double.compare(b.corrMean(), a.corrMean());
});
CnnEval bestE99 = evalsE99.get(0);
CnnTuneParams bestParamsE99 = bestE99.params();
System.out.println("Total unique E99 candidates evaluated: " + evalsE99.size());
System.out.printf(Locale.US,
    "E99 Stage B (refined) best: baseBlend=%.2f, gamma=%.2f, edgeBoost=%.2f, unsharp=%.2f | sep=%.5f corr=%.5f score=%.5f%n",
    bestParamsE99.baseBlend(), bestParamsE99.gamma(), bestParamsE99.edgeBoost(), bestParamsE99.unsharpAmount(),
    bestE99.sepMean(), bestE99.corrMean(), bestE99.score()
 );

double scoreDeltaE99vsR40 = bestE99.score() - bestCnn.score();
System.out.printf(Locale.US, "Delta (best E99 - best Rank40) = %.5f%n", scoreDeltaE99vsR40);

// 3) Score all top-3 methods using the selected CNN params
Map<String, double[]> scoreSums = new HashMap<>();
for (MethodCfg m : methods) scoreSums.put(m.key(), new double[] {0.0, 0.0, 0.0}); // sep, corr, count

for (TileDatum t : tiles) {
    for (MethodCfg method : methods) {
        double[][] map = "cnn_rank40".equals(method.key())
            ? cnnMapFromFeatures(t.cnnFeat(), 40, bestParams)
            : methodMap(t.g(), method.key(), bestParams).map();
        double[][] map01 = normalize01(map);

        Path outImg = t.dir().resolve(t.base() + "_tile" + t.id() + "_" + method.key() + ".jpg");
        ImageIO.write(toGrayImage01(map), "jpg", outImg.toFile());

        double[] io = inOutMeans(map01, t.maskG());
        double sep = io[0] - io[1];
        double corr = pearson2D(map01, t.rec01());

        double[] s = scoreSums.get(method.key());
        s[0] += sep;
        s[1] += corr;
        s[2] += 1.0;
    }
}

System.out.println("Completed per-tile outputs for datasets: " + bases);

List<MethodScore> scores = new ArrayList<>();
for (MethodCfg m : methods) {
    double[] s = scoreSums.get(m.key());
    int n = (int)Math.round(s[2]);
    if (n == 0) continue;
    double sepMean = s[0] / n;
    double corrMean = s[1] / n;
    double score = 0.7 * sepMean + 0.3 * corrMean;
    scores.add(new MethodScore(m.label(), sepMean, corrMean, score, n));
}
scores.sort((a, b) -> Double.compare(b.score(), a.score()));

System.out.println("\nTop-3 method ranking (higher is better)");
System.out.println("score = 0.7*(mask-separation) + 0.3*(corr-with-rec)");
System.out.println("method\tsepMean\tcorrMean\tscore\tnTiles");
for (MethodScore s : scores) {
    System.out.printf(Locale.US, "%s\t%.5f\t%.5f\t%.5f\t%d%n", s.label(), s.sepMean(), s.corrMean(), s.score(), s.nTiles());
}

if (!scores.isEmpty()) {
    MethodScore best = scores.get(0);
    System.out.printf(Locale.US, "\nBest method: %s (score=%.5f)%n", best.label(), best.score());
}

Path scoreCsv = root.resolve("notebooks").resolve("_assets").resolve("fiba_wavelet_montage").resolve("graph_laplacian_svd_scores_top3.csv");
StringBuilder sb = new StringBuilder();
sb.append("method,sepMean,corrMean,score,nTiles\n");
for (MethodScore s : scores) {
    sb.append("\"").append(s.label()).append("\",")
      .append(String.format(Locale.US, "%.6f,%.6f,%.6f,%d\n", s.sepMean(), s.corrMean(), s.score(), s.nTiles()));
}
Files.writeString(scoreCsv, sb.toString(), StandardCharsets.UTF_8);
System.out.println("Scores written: " + scoreCsv);

Loaded tiles for benchmarking: 20
Stage A (coarse) best: baseBlend=0.55 gamma=0.95 edgeBoost=0.10 unsharp=0.12 | score=0.21125
Total unique CNN candidates evaluated: 96
Stage B (refined) best: baseBlend=0.58, gamma=0.92, edgeBoost=0.06, unsharp=0.08 | sep=0.03985 corr=0.61997 score=0.21389
E99 Stage A (coarse) best: baseBlend=0.55 gamma=0.95 edgeBoost=0.10 unsharp=0.12 | score=0.21100
Total unique E99 candidates evaluated: 96
E99 Stage B (refined) best: baseBlend=0.52, gamma=0.92, edgeBoost=0.06, unsharp=0.08 | sep=0.03998 corr=0.61905 score=0.21370
Delta (best E99 - best Rank40) = -0.00019
Completed per-tile outputs for datasets: [C15D5P001_1, Picture1]

Top-3 method ranking (higher is better)
score = 0.7*(mask-separation) + 0.3*(corr-with-rec)
method	sepMean	corrMean	score	nTiles
SVD Energy-99%	0.02416	0.87208	0.27853	20
SVD Rank-20	0.02391	0.85827	0.27422	20
CNN-style ConvBank + SVD Rank-40	0.03985	0.61997	0.21389	20

Best method: SVD Energy-99% (score=0.27853)
Scores written: c:\Us

In [30]:
// ---------- Build side-by-side comparison stack panels (top-3 only) ----------
BufferedImage fitHeight(BufferedImage src, int targetH) {
    int targetW = Math.max(1, (int)Math.round(src.getWidth() * (targetH / (double)src.getHeight())));
    BufferedImage out = new BufferedImage(targetW, targetH, BufferedImage.TYPE_INT_RGB);
    Graphics2D g = out.createGraphics();
    g.setRenderingHint(RenderingHints.KEY_INTERPOLATION, RenderingHints.VALUE_INTERPOLATION_BILINEAR);
    g.drawImage(src, 0, 0, targetW, targetH, null);
    g.dispose();
    return out;
}

BufferedImage buildMethodStack(Path dir, String base, List<Integer> tileIds, String methodKey) throws Exception {
    List<BufferedImage> imgs = new ArrayList<>();
    int maxW = 1;
    for (int id : tileIds) {
        Path p = dir.resolve(base + "_tile" + id + "_" + methodKey + ".jpg");
        if (!Files.isRegularFile(p)) continue;
        BufferedImage img = ImageIO.read(p.toFile());
        if (img == null) continue;
        imgs.add(img);
        maxW = Math.max(maxW, img.getWidth());
    }
    if (imgs.isEmpty()) throw new RuntimeException("No method tiles found for " + methodKey);

    int pad = 6;
    int totalH = pad;
    for (BufferedImage bi : imgs) totalH += bi.getHeight() + pad;
    BufferedImage stack = new BufferedImage(maxW + 2 * pad, totalH, BufferedImage.TYPE_INT_RGB);
    Graphics2D g = stack.createGraphics();
    g.setColor(Color.WHITE);
    g.fillRect(0, 0, stack.getWidth(), stack.getHeight());
    int y = pad;
    for (BufferedImage bi : imgs) {
        int x = (stack.getWidth() - bi.getWidth()) / 2;
        g.drawImage(bi, x, y, null);
        y += bi.getHeight() + pad;
    }
    g.dispose();
    return stack;
}

Path root2 = findProjectRoot(Paths.get(System.getProperty("user.dir")));
List<String> bases2 = Arrays.asList("C15D5P001_1", "Picture1");
List<Path> dirs2 = Arrays.asList(
    root2.resolve("notebooks").resolve("_assets").resolve("fiba_wavelet_montage").resolve("generated_from_source"),
    root2.resolve("notebooks").resolve("_assets").resolve("fiba_wavelet_montage").resolve("generated_from_source_picture1")
);

List<MethodCfg> methods2 = Arrays.asList(
    new MethodCfg("SVD Rank-20", "svd_rank20"),
    new MethodCfg("SVD Energy-99%", "svd_energy99"),
    new MethodCfg("CNN-style ConvBank + SVD Rank-40", "cnn_rank40")
);

for (int ds = 0; ds < bases2.size(); ds++) {
    String base = bases2.get(ds);
    Path dir = dirs2.get(ds);

    Pattern p = Pattern.compile("^" + Pattern.quote(base) + "_tile(\\d+)_crop\\.jpg$");
    List<Integer> ids = new ArrayList<>();
    try (DirectoryStream<Path> stream = Files.newDirectoryStream(dir, "*_crop.jpg")) {
        for (Path f : stream) {
            Matcher m = p.matcher(f.getFileName().toString());
            if (m.matches()) ids.add(Integer.parseInt(m.group(1)));
        }
    }
    ids.sort(Comparator.naturalOrder());
    List<Integer> firstTen = ids.stream().filter(i -> i >= 1 && i <= 10).toList();
    if (firstTen.isEmpty()) firstTen = ids;

    List<BufferedImage> cols = new ArrayList<>();
    List<String> labels = new ArrayList<>();
    int maxColH = 1;

    for (MethodCfg m : methods2) {
        BufferedImage st = buildMethodStack(dir, base, firstTen, m.key());
        cols.add(st);
        labels.add(m.label());
        maxColH = Math.max(maxColH, st.getHeight());
    }

    List<BufferedImage> resized = new ArrayList<>();
    int totalW = 32;
    int gap = 12;
    for (BufferedImage c : cols) {
        BufferedImage r = fitHeight(c, maxColH);
        resized.add(r);
        totalW += r.getWidth() + gap;
    }

    int titleH = 84, labelH = 36, padBottom = 16;
    BufferedImage panel = new BufferedImage(totalW, titleH + labelH + maxColH + padBottom, BufferedImage.TYPE_INT_RGB);
    Graphics2D g = panel.createGraphics();
    g.setRenderingHint(RenderingHints.KEY_ANTIALIASING, RenderingHints.VALUE_ANTIALIAS_ON);
    g.setColor(Color.WHITE);
    g.fillRect(0, 0, panel.getWidth(), panel.getHeight());

    g.setColor(new Color(12, 33, 64));
    g.setFont(new Font("SansSerif", Font.BOLD, 22));
    g.drawString("Top-3 model comparison: " + base, 14, 30);
    g.setFont(new Font("SansSerif", Font.PLAIN, 12));
    g.drawString("SVD rank-20 vs SVD energy-99% vs CNN-style high-rank baseline", 14, 50);

    int x = 14;
    for (int i = 0; i < resized.size(); i++) {
        BufferedImage c = resized.get(i);
        g.setColor(new Color(25, 25, 25));
        g.setFont(new Font("SansSerif", Font.BOLD, 11));
        g.drawString(labels.get(i), x + 2, titleH + 14);
        g.drawImage(c, x, titleH + labelH, null);
        x += c.getWidth() + gap;
    }
    g.dispose();

    Path out = dir.resolve(base + "_top3_svd_cnn_compare.png");
    ImageIO.write(panel, "png", out.toFile());
    System.out.println("Comparison panel written: " + out);
}

System.out.println("Panel generation complete ✅");

Comparison panel written: c:\Users\dunnmk\repos\imgjplugin\notebooks\_assets\fiba_wavelet_montage\generated_from_source\C15D5P001_1_top3_svd_cnn_compare.png
Comparison panel written: c:\Users\dunnmk\repos\imgjplugin\notebooks\_assets\fiba_wavelet_montage\generated_from_source_picture1\Picture1_top3_svd_cnn_compare.png
Panel generation complete ✅


## Comparison outputs (top-3 only)

Dataset A panel:
![](./_assets/fiba_wavelet_montage/generated_from_source/C15D5P001_1_top3_svd_cnn_compare.png)

Dataset B panel:
![](./_assets/fiba_wavelet_montage/generated_from_source_picture1/Picture1_top3_svd_cnn_compare.png)

**Latest (last) generated figure embed:**
![](./_assets/fiba_wavelet_montage/generated_from_source_picture1/Picture1_top3_svd_cnn_compare.png)

Top-3 ranking table:
- `notebooks/_assets/fiba_wavelet_montage/graph_laplacian_svd_scores_top3.csv`

## Notes
- Current top-3 comparison is still reported as **SVD rank-20**, **SVD energy-99%**, and **CNN-style conv-bank + SVD rank-40** for consistency of output figures and CSV.
- The CNN pipeline is now explicitly parameterized by $\theta=(\alpha,\gamma,\eta,\lambda)$ where: $\alpha$=`baseBlend`, $\gamma$=`gamma`, $\eta$=`edgeBoost`, $\lambda$=`unsharpAmount`.
- Cached evaluation reuses deterministic conv channels $(G_3,G_5,L,D,T,B,Z)$; this preserves objective values while reducing repeated convolution cost during tuning.
- Two-stage tuning (coarse then local refinement) is run independently for rank-40 and energy-99 CNN projections, producing a direct score delta $\Delta = J_{\text{E99}}-J_{\text{R40}}$.
- If you want next, we can expose the full candidate table $\{\theta,\mathrm{sep},\mathrm{corr},J\}$ as a CSV for both CNN heads to analyze parameter sensitivity.

## Unified trajectory-angle objective across FFT, Wavelet, and SVD

This section applies one shared objective to all methods:

$$
\text{score}=0.7\cdot(\text{mask-separation})+0.3\cdot(\text{corr-with-rec})
$$

The mask is not reused from pipeline outputs. Instead, each tile gets a **trajectory-angle mask** estimated from the tile itself (time-averaged single image assumption), then all methods are scored against that same directional mask logic.

In [34]:
import java.awt.image.BufferedImage;
import javax.imageio.ImageIO;
import java.nio.file.*;
import java.util.*;
import java.util.regex.*;

record UniMethod(String label) {}
record UniScore(String label, double sepMean, double corrMean, double score, double meanThetaDeg, int nTiles) {}

double[][] uniGray01(BufferedImage img) {
    int h = img.getHeight(), w = img.getWidth();
    double[][] out = new double[h][w];
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            int rgb = img.getRGB(x, y);
            int r = (rgb >> 16) & 0xff;
            int g = (rgb >> 8) & 0xff;
            int b = rgb & 0xff;
            out[y][x] = (0.299 * r + 0.587 * g + 0.114 * b) / 255.0;
        }
    }
    return out;
}

double[][] uniNormalize01(double[][] a) {
    int h = a.length, w = a[0].length;
    double min = Double.POSITIVE_INFINITY, max = Double.NEGATIVE_INFINITY;
    for (double[] row : a) for (double v : row) { min = Math.min(min, v); max = Math.max(max, v); }
    double span = Math.max(1e-9, max - min);
    double[][] out = new double[h][w];
    for (int y = 0; y < h; y++) for (int x = 0; x < w; x++) out[y][x] = (a[y][x] - min) / span;
    return out;
}

double[][] uniConvSame(double[][] src, double[][] k) {
    int h = src.length, w = src[0].length;
    int kh = k.length, kw = k[0].length;
    int ry = kh / 2, rx = kw / 2;
    double[][] out = new double[h][w];
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            double s = 0.0;
            for (int j = -ry; j <= ry; j++) {
                int yy = Math.max(0, Math.min(h - 1, y + j));
                for (int i = -rx; i <= rx; i++) {
                    int xx = Math.max(0, Math.min(w - 1, x + i));
                    s += src[yy][xx] * k[j + ry][i + rx];
                }
            }
            out[y][x] = s;
        }
    }
    return out;
}

double[][] uniGaborKernel(int size, double sigma, double theta, double lambda, double gamma, double psi) {
    int r = size / 2;
    double[][] k = new double[size][size];
    for (int y = -r; y <= r; y++) {
        for (int x = -r; x <= r; x++) {
            double xr = x * Math.cos(theta) + y * Math.sin(theta);
            double yr = -x * Math.sin(theta) + y * Math.cos(theta);
            double gauss = Math.exp(-(xr * xr + (gamma * gamma) * yr * yr) / (2.0 * sigma * sigma));
            double wave = Math.cos(2.0 * Math.PI * xr / lambda + psi);
            k[y + r][x + r] = gauss * wave;
        }
    }
    return k;
}

double[][] uniGaborEnergy(double[][] src) {
    int h = src.length, w = src[0].length;
    int ori = 8, kSize = 13;
    double sigma = 2.2, lambda = 5.5, gamma = 0.65;
    double[][] out = new double[h][w];
    for (int oi = 0; oi < ori; oi++) {
        double theta = (Math.PI * oi) / ori;
        double[][] kRe = uniGaborKernel(kSize, sigma, theta, lambda, gamma, 0.0);
        double[][] kIm = uniGaborKernel(kSize, sigma, theta, lambda, gamma, Math.PI / 2.0);
        double[][] re = uniConvSame(src, kRe);
        double[][] im = uniConvSame(src, kIm);
        for (int y = 0; y < h; y++) {
            for (int x = 0; x < w; x++) {
                double mag = Math.sqrt(re[y][x] * re[y][x] + im[y][x] * im[y][x]);
                if (mag > out[y][x]) out[y][x] = mag;
            }
        }
    }
    return out;
}

double[][] uniMorletEnergy(double[][] src) {
    return uniGaborEnergy(src);
}

double[][] uniChirpletEnergy(double[][] src) {
    int h = src.length, w = src[0].length;
    int ori = 8, size = 13;
    double sigma = 2.2, lambda = 5.5, gamma = 0.65, chirp = 0.015;
    int r = size / 2;
    double[][] out = new double[h][w];
    for (int oi = 0; oi < ori; oi++) {
        double theta = (Math.PI * oi) / ori;
        double[][] k = new double[size][size];
        for (int y = -r; y <= r; y++) {
            for (int x = -r; x <= r; x++) {
                double xr = x * Math.cos(theta) + y * Math.sin(theta);
                double yr = -x * Math.sin(theta) + y * Math.cos(theta);
                double gauss = Math.exp(-(xr * xr + (gamma * gamma) * yr * yr) / (2.0 * sigma * sigma));
                double phase = 2.0 * Math.PI * (xr / lambda + chirp * xr * xr);
                k[y + r][x + r] = gauss * Math.cos(phase);
            }
        }
        double[][] resp = uniConvSame(src, k);
        for (int y = 0; y < h; y++) for (int x = 0; x < w; x++) out[y][x] = Math.max(out[y][x], Math.abs(resp[y][x]));
    }
    return out;
}

double[][] uniSobelX(double[][] src) {
    double[][] kx = new double[][] {{-1,0,1},{-2,0,2},{-1,0,1}};
    return uniConvSame(src, kx);
}

double[][] uniSobelY(double[][] src) {
    double[][] ky = new double[][] {{-1,-2,-1},{0,0,0},{1,2,1}};
    return uniConvSame(src, ky);
}

double uniDominantTheta(double[][] img01) {
    double[][] gx = uniSobelX(img01);
    double[][] gy = uniSobelY(img01);
    int h = img01.length, w = img01[0].length;
    double jxx = 0.0, jyy = 0.0, jxy = 0.0;
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            jxx += gx[y][x] * gx[y][x];
            jyy += gy[y][x] * gy[y][x];
            jxy += gx[y][x] * gy[y][x];
        }
    }
    return 0.5 * Math.atan2(2.0 * jxy, jxx - jyy);
}

double[][] uniTrajectoryMask(int h, int w, double theta) {
    double[][] m = new double[h][w];
    double cx = (w - 1) / 2.0;
    double cy = (h - 1) / 2.0;
    double sigmaPerp = 0.10 * Math.min(h, w);
    double sigmaAlong = 0.45 * Math.min(h, w);
    double twoPerp = 2.0 * sigmaPerp * sigmaPerp;
    double twoAlong = 2.0 * sigmaAlong * sigmaAlong;

    double c = Math.cos(theta), s = Math.sin(theta);
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            double dx = x - cx;
            double dy = y - cy;
            double along = dx * c + dy * s;
            double perp = -dx * s + dy * c;
            m[y][x] = Math.exp(-(perp * perp) / twoPerp) * Math.exp(-(along * along) / twoAlong);
        }
    }
    return uniNormalize01(m);
}

double uniMaskSeparation(double[][] map01, double[][] trajMask01, double thr) {
    int h = Math.min(map01.length, trajMask01.length);
    int w = Math.min(map01[0].length, trajMask01[0].length);
    double in = 0.0, out = 0.0;
    int nin = 0, nout = 0;
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            if (trajMask01[y][x] >= thr) { in += map01[y][x]; nin++; }
            else { out += map01[y][x]; nout++; }
        }
    }
    double inMean = nin == 0 ? 0.0 : in / nin;
    double outMean = nout == 0 ? 0.0 : out / nout;
    double sep = inMean - outMean;
    return Math.max(0.0, Math.min(1.0, sep));
}

double uniPearsonCenter(double[][] a, double[][] b) {
    int h = Math.min(a.length, b.length);
    int w = Math.min(a[0].length, b[0].length);
    int y0 = h / 4, y1 = h - h / 4;
    int x0 = w / 4, x1 = w - w / 4;
    int n = 0;
    double sa = 0.0, sb = 0.0;
    for (int y = y0; y < y1; y++) for (int x = x0; x < x1; x++) { sa += a[y][x]; sb += b[y][x]; n++; }
    if (n < 2) return 0.0;
    double ma = sa / n, mb = sb / n;
    double num = 0.0, da = 0.0, db = 0.0;
    for (int y = y0; y < y1; y++) {
        for (int x = x0; x < x1; x++) {
            double xa = a[y][x] - ma;
            double xb = b[y][x] - mb;
            num += xa * xb;
            da += xa * xa;
            db += xb * xb;
        }
    }
    return num / Math.sqrt(Math.max(1e-12, da * db));
}

double[][] uniMethodMap(String label, double[][] crop01, Path fftPath) throws Exception {
    if ("FFT Pipeline".equals(label)) {
        BufferedImage f = ImageIO.read(fftPath.toFile());
        if (f == null) throw new RuntimeException("Could not read FFT map: " + fftPath);
        return uniGray01(f);
    }
    if ("Gabor Transform".equals(label)) return uniNormalize01(uniGaborEnergy(crop01));
    if ("Chirplet Transform".equals(label)) return uniNormalize01(uniChirpletEnergy(crop01));
    if ("Morlet Wavelet".equals(label)) return uniNormalize01(uniMorletEnergy(crop01));
    if ("SVD Rank-20".equals(label)) return uniNormalize01(svdLowRank(crop01, 20, null, 60).recon());
    if ("SVD Energy-99%".equals(label)) return uniNormalize01(svdLowRank(crop01, null, 0.99, 60).recon());
    throw new IllegalArgumentException("Unknown label: " + label);
}

Path rootUni = findProjectRoot(Paths.get(System.getProperty("user.dir")));
if (rootUni == null) throw new RuntimeException("Could not locate project root");

List<String> basesUni = Arrays.asList("C15D5P001_1", "Picture1");
List<Path> dirsWaveUni = Arrays.asList(
    rootUni.resolve("notebooks").resolve("_assets").resolve("fiba_wavelet_montage").resolve("generated_from_source"),
    rootUni.resolve("notebooks").resolve("_assets").resolve("fiba_wavelet_montage").resolve("generated_from_source_picture1")
);
List<Path> dirsFftUni = Arrays.asList(
    rootUni.resolve("notebooks").resolve("_assets").resolve("fiba_tile_montage").resolve("generated_from_source"),
    rootUni.resolve("notebooks").resolve("_assets").resolve("fiba_tile_montage").resolve("generated_from_source_picture1")
);

List<UniMethod> methodsUni = Arrays.asList(
    new UniMethod("FFT Pipeline"),
    new UniMethod("Gabor Transform"),
    new UniMethod("Chirplet Transform"),
    new UniMethod("Morlet Wavelet"),
    new UniMethod("SVD Rank-20"),
    new UniMethod("SVD Energy-99%")
);

Map<String, double[]> agg = new LinkedHashMap<>(); // sep, corr, theta, n
for (UniMethod m : methodsUni) agg.put(m.label(), new double[] {0.0, 0.0, 0.0, 0.0});

final double TRAJ_THR = 0.55;

for (int ds = 0; ds < basesUni.size(); ds++) {
    String base = basesUni.get(ds);
    Path wDir = dirsWaveUni.get(ds);
    Path fDir = dirsFftUni.get(ds);

    Pattern p = Pattern.compile("^" + Pattern.quote(base) + "_tile(\\d+)_crop\\.jpg$");
    List<Integer> ids = new ArrayList<>();
    try (DirectoryStream<Path> stream = Files.newDirectoryStream(wDir, "*_crop.jpg")) {
        for (Path f : stream) {
            Matcher m = p.matcher(f.getFileName().toString());
            if (m.matches()) ids.add(Integer.parseInt(m.group(1)));
        }
    }
    ids.sort(Comparator.naturalOrder());

    for (int id : ids) {
        Path cropPath = wDir.resolve(base + "_tile" + id + "_crop.jpg");
        Path recPath = wDir.resolve(base + "_tile" + id + "_rec.jpg");
        Path fftPath = fDir.resolve(base + "_tile" + id + "_fft.jpg");
        if (!Files.isRegularFile(cropPath) || !Files.isRegularFile(recPath) || !Files.isRegularFile(fftPath)) continue;

        BufferedImage cropBI = ImageIO.read(cropPath.toFile());
        BufferedImage recBI = ImageIO.read(recPath.toFile());
        if (cropBI == null || recBI == null) continue;

        double[][] crop01 = uniGray01(cropBI);
        double[][] rec01 = uniGray01(recBI);

        double theta = uniDominantTheta(crop01);
        double[][] trajMask = uniTrajectoryMask(crop01.length, crop01[0].length, theta);

        for (UniMethod method : methodsUni) {
            double[][] map01 = uniMethodMap(method.label(), crop01, fftPath);
            double sep = uniMaskSeparation(map01, trajMask, TRAJ_THR);
            double corrRaw = uniPearsonCenter(map01, rec01);
            double corr = Math.max(0.0, Math.min(1.0, 0.5 * (corrRaw + 1.0)));

            double[] a = agg.get(method.label());
            a[0] += sep;
            a[1] += corr;
            a[2] += Math.toDegrees(theta);
            a[3] += 1.0;
        }
    }
}

List<UniScore> finalScores = new ArrayList<>();
for (UniMethod m : methodsUni) {
    double[] a = agg.get(m.label());
    int n = (int)Math.round(a[3]);
    if (n == 0) continue;
    double sepMean = a[0] / n;
    double corrMean = a[1] / n;
    double score = 0.7 * sepMean + 0.3 * corrMean;
    double thetaMean = a[2] / n;
    finalScores.add(new UniScore(m.label(), sepMean, corrMean, score, thetaMean, n));
}

finalScores.sort((u, v) -> Double.compare(v.score(), u.score()));

System.out.println("Unified trajectory-angle scoring across FFT + Wavelet + SVD");
System.out.println("score = 0.7*(mask-separation) + 0.3*(corr-with-rec)");
System.out.println("mask = trajectory-angle mask from each tile (single time-averaged image assumption)");
System.out.println("label\tsepMean\tcorrMean\tscore\tmeanThetaDeg\tnTiles");
for (UniScore s : finalScores) {
    System.out.printf(Locale.US, "%s\t%.5f\t%.5f\t%.5f\t%.2f\t%d%n", s.label(), s.sepMean(), s.corrMean(), s.score(), s.meanThetaDeg(), s.nTiles());
}

if (!finalScores.isEmpty()) {
    UniScore best = finalScores.get(0);
    System.out.printf(Locale.US, "\nBest unified method: %s (score=%.5f)%n", best.label(), best.score());
}

System.out.println("Unified cross-method trajectory scoring complete ✅");

Unified trajectory-angle scoring across FFT + Wavelet + SVD
score = 0.7*(mask-separation) + 0.3*(corr-with-rec)
mask = trajectory-angle mask from each tile (single time-averaged image assumption)
label	sepMean	corrMean	score	meanThetaDeg	nTiles
FFT Pipeline	0.28923	0.51113	0.35580	34.63	20
SVD Energy-99%	0.03733	0.96059	0.31431	34.63	20
SVD Rank-20	0.03647	0.95333	0.31153	34.63	20
Gabor Transform	0.06898	0.87424	0.31056	34.63	20
Morlet Wavelet	0.06898	0.87424	0.31056	34.63	20
Chirplet Transform	0.04914	0.91225	0.30808	34.63	20

Best unified method: FFT Pipeline (score=0.35580)
Unified cross-method trajectory scoring complete ✅
